## Coding a Transformer from scratch using PyTorch

<img src="images/attention-is-all-you-need-ModalNet-21.png" width="400" height="589">

Attention is All You Need: https://doi.org/10.48550/arXiv.1706.03762

I followed this tutorial by Umar Jamil: https://www.youtube.com/watch?v=bCz4OMemCcA

Code from the tutorial: https://github.com/hkproj/pytorch-transformer/blob/main/model.py

In [61]:
import torch
import torch.nn as nn
import math

In [62]:
class InputEmbeddings(nn.Module):
    #
    # docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html
    #
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model    = d_model
        self.vocab_size = vocab_size
        self.embedding  = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)# sqrt(dim) from paper

In [63]:
class PositionalEmbeddings(nn.Module):
    #
    # docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html
    #
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Matrix: [seq_len, d_model]
        self.pe = torch.zeros(seq_len, d_model)

        # Position Embedding Vector functions PE(pos,2i) and PE(pos,2i+1): [seq_len, 1] (3.5 p. 6)
        numerator_pos = torch.arange(0, seq_len, dtype=torch.float)
        numerator_pos = torch.unsqueeze(numerator_pos, 1)# at tensor index 1: Returns a new tensor with a dimension of size one inserted at the specified position.
        # Modified log space denominator
        denominator = torch.exp( torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model) )

        # Posistional Embedding (3.5 p. 6)
        self.pe[:, 0::2] = torch.sin( numerator_pos / denominator)
        self.pe[:, 1::2] = torch.cos( numerator_pos / denominator)

        # Add a batch index: [1, seq_len, d_model]
        self.pe = self.pe.unsqueeze(0)

        # Register PE as a part of the module's state (but not as a parameter)
        # PositionalEmbeddings.get_buffer('Positional Embeddings')
        self.register_buffer('Positional Embeddings', self.pe)

    def forward(self, x):
        # x can be shorter than pe and requires gradient attribute set to False (pe is frozen)
        x = x + ( self.pe[:, :x.shape[1], :] ).requires_grad_(False)
        return x

In [64]:
class LayerNormalization(nn.Module):
    #
    # docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html
    #
    def __init__(self, epsilon:float = 10**-6) -> None:
        super().__init__()
        self.epsilon = epsilon
        self.gamma   = nn.Parameter( torch.ones(1)   )# multiplicative identity
        self.bias    = nn.Parameter( torch.zeros(1) )# additive identity, "intercept"

    def forward(self, x: torch.Tensor):
        # 
        mean        = x.mean(dim=-1, keepdim=True )
        std         = x.std( dim=-1, keepdim=True )
        numerator   = x - mean
        denominator = torch.sqrt( std**2 + self.epsilon)
        x_hat       = numerator / denominator
        # Multiply by gamma, add bias
        x_hat       = self.gamma* x_hat + self.bias
        return x_hat

In [65]:
# 3.3 Position-wise Feed-Forward Networks (p.5)
class FeedForward(nn.Module):
    #
    # docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html
    #
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)# W1 + b1
        self.dropout  = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)# W2 + b2

    def forward(self, x):
        # [batch, seq_len, d_model] --> [batch, seq_len, d_ff] --> [batch, seq_len, d_model]
        # layer1 = self.linear_1(x)
        # layer2 = torch.relu(layer1)
        # layer3 = self.dropout(layer2)
        # layer4 = self.linear_2(layer3)
        return self.linear_2( self.dropout( torch.relu( self.linear_1(x) )))

In [66]:
# 3.2.2 Multi-Head Attention (p.4-5)
class MultiHeadAttention(nn.Module):
    #
    # docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html
    #
    def __init__(self, d_model: int, heads: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.heads   = heads
        assert (d_model % heads == 0), "Error in MultiHeadAttention module: d_model is not divisible by the number of attention heads"

        # d_k = d_v = d_model / h(eads) = 64 (p.5)
        self.d_k = d_model // heads#  where // outputs a floored int vs / returns a float

        # Q, K, V
        self.w_q     = nn.Linear(d_model, d_model)
        self.w_k     = nn.Linear(d_model, d_model)
        self.w_v     = nn.Linear(d_model, d_model)
        self.w_o     = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]# if we don't instantiate the class we can't access self.d_k

        # 3.2.1 Scaled Dot-Product Attention (p.3-4)
        # [batch, heads, seq_len, d_k] --> [batch, heads, seq_len, seq_len]
        dot_product_scaled = ( query @ key.transpose(-2, -1) ) / math.sqrt(d_k)
        
        if mask is not None:
            # docs.pytorch.org/docs/2.14/generated/torch.Tensor.masked_fill_.html
            dot_product_scaled.masked_fill_(mask == 0, -1e9)
        
        attention_scores = dot_product_scaled.softmax(dim=-1)# [batch, heads, seq_len, seq_len]
        
        if dropout is not None:
            attention_scores = dropout(attention_scores)

        scaled_dot_product_attention = attention_scores @ value

        # also return raw attention_scores for visualization
        return scaled_dot_product_attention, attention_scores

    def split_heads(self, x):
        return x.view(x.size(0), x.size(1), self.heads, self.d_k).transpose(1, 2)
 
    def forward(self, q, k, v, mask):
        # All three: [batch, seq_len, d_model]
        print("----- DEBUG ----- MultiHeadAttention.forward: query tensor q.shape:", q.shape)
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)

        # Reshape using .view(), and interchange seq_len with self.heads
        # [batch, seq_len, d_model] --> [batch, seq_len, heads, d_k] --> [batch, heads, seq_len, d_k]
        #query = query.view( query.shape[0], query.shape[1], self.heads, self.d_k).transpose(1, 2)
        #key   = key.view(   key.shape[0],   key.shape[1],   self.heads, self.d_k).transpose(1, 2)
        #value = value.view( value.shape[0], value.shape[1], self.heads, self.d_k).transpose(1, 2)
        query  = self.split_heads(query)
        key    = self.split_heads(key)
        value  = self.split_heads(value)

        x, self.attention_scores = self.attention(query, key, value, mask, self.dropout)

        # [batch, heads, seq_len, d_k] --> [batch, seq_len, heads, d_k] --concat--> [batch, seq_len, d_model] undoing what we did just above
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.d_model)
        
        return self.w_o(x)
        

In [67]:
class SkipConnection(nn.Module):
    def __init__(self, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm    = LayerNormalization()

    def forward(self, x, sublayer):
        return  x + self.dropout( sublayer( self.norm(x) ))# note normalization before passing x to sublayer

In [68]:
class EncoderBlock(nn.Module):
    def __init__(self, multihead_attention: MultiHeadAttention, feed_forward: FeedForward, dropout: float) -> None:
        super().__init__()
        self.multihead_attention = multihead_attention
        self.feed_forward         = feed_forward
        self.skip_connections    = nn.ModuleList( [SkipConnection(dropout) for _ in range(2)] )

    def forward(self, x, input_mask):
        x = self.skip_connections[0](x, lambda x: self.multihead_attention(x,x,x, input_mask))# (x, sublayer)
        x = self.skip_connections[1](x, self.feed_forward)# (x, sublayer)
        return x

In [69]:
class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm   = LayerNormalization()

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)# forward of EncoderBlock
        return self.norm(x)

In [70]:
class DecoderBlock(nn.Module):
    def __init__(self, masked_attention: MultiHeadAttention, cross_attention: MultiHeadAttention, feed_forward: FeedForward, dropout: float) -> None:
        super().__init__()
        self.masked_attention = masked_attention
        self.cross_attention  = cross_attention
        self.feed_forward     = feed_forward
        # no dropout
        self.skip_connections = nn.ModuleList( [SkipConnection(dropout) for _ in range(3)] )

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.skip_connections[0](x, lambda x: self.masked_attention(x,x,x, tgt_mask) )
        x = self.skip_connections[1](x, lambda x: self.cross_attention(x, encoder_output, encoder_output, src_mask) )
        x = self.skip_connections[2](x, self.feed_forward)
        return x

In [71]:
class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm   = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

In [72]:
class LinearProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return torch.log_softmax(self.proj(x), dim=-1)

In [73]:
class Transformer(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEmbeddings, tgt_pos: PositionalEmbeddings, projection_layer: LinearProjectionLayer):
        super().__init__()
        self.encoder   = encoder
        self.decoder   = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos   = src_pos
        self.tgt_pos   = tgt_pos
        self.proj      = projection_layer

    def encode(self, src, src_mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        print("----- DEBUG ----- Transformer.encode embed + positionally encoded src.shape:", src.shape)
        return self.encoder(src, src_mask)

    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        print("----- DEBUG ----- Transformer.decode embed + positionally encoded tgt.shape:", tgt.shape )
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)# forward of the Decoder

    def project(self, x):
        return self.proj(x)

    def forward(self, src_input, tgt_input):
        src_mask = None
        tgt_mask = None
        print("----- DEBUG ----- Transformer: masks OK.")
        print("----- DEBUG ----- Transformer: entering encoder...")
        encoder_output = self.encode(src_input, src_mask)
        print("----- DEBUG ----- Transformer: encode OK.")
        print("----- DEBUG ----- Transformer: encoder_output.shape:", encoder_output.shape)
        print("\n")
        print("----- DEBUG ----- Transformer: entering decoder...")
        output  = self.decode(encoder_output, src_mask, tgt_input, tgt_mask)
        print("----- DEBUG ----- Transformer: decode OK.")
        return output


In [74]:
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int = 512, N: int = 6, heads: int = 8, dropout: float = 0.1, d_ff: int = 2048) -> Transformer:
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)
    src_pos   = PositionalEmbeddings(d_model, src_seq_len, dropout)
    tgt_pos   = PositionalEmbeddings(d_model, tgt_seq_len, dropout)

    encoder_blocks = []
    for _ in range(N):
        encoder_self_attention_block = MultiHeadAttention(d_model, heads, dropout)
        feed_forward_block           = FeedForward(d_model, d_ff, dropout)
        encoder_block                = EncoderBlock(encoder_self_attention_block, feed_forward_block, dropout)
        encoder_blocks.append(encoder_block)

    decoder_blocks = []
    for _ in range(N):
        decoder_masked_attention_block = MultiHeadAttention(d_model, heads, dropout)
        decoder_cross_attention_block  = MultiHeadAttention(d_model, heads, dropout)
        feed_forward_block             = FeedForward(d_model, d_ff, dropout)
        decoder_block                  = DecoderBlock(decoder_masked_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
        decoder_blocks.append(decoder_block)

    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))
    projection_layer = LinearProjectionLayer(d_model, tgt_vocab_size)
    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)

    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return transformer

## Test the Transformer

In [75]:
batch_size = 2
src_vocab_size = 1000
tgt_vocab_size = 1000
src_seq_len = 10
tgt_seq_len = 12
d_model = 512
num_tokens = 1000
N = 6
heads = 8
dropout = 0.1
d_ff = 2048

src_input = torch.randint(0, num_tokens, (batch_size, src_seq_len))
tgt_input = torch.randint(0, num_tokens, (batch_size, tgt_seq_len))
print("Input into transformer, source:\n", src_input)
print("Input into transformer, target:\n", tgt_input)

model = build_transformer(src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len, d_model, N, heads, dropout, d_ff)

print("Transformer model:\n", model)

with torch.no_grad():
    output = model(src_input, tgt_input)

print("Transformer output using random inputs:\n", output)

Input into transformer, source:
 tensor([[978, 939, 539, 412, 167, 158, 787, 997, 469,  81],
        [843, 261, 206, 336, 825, 372, 251, 725, 775, 648]])
Input into transformer, target:
 tensor([[787,  79, 205,  62, 609, 857, 829, 371, 285, 191, 629, 615],
        [589, 259, 853, 797, 144, 184, 946, 580, 684, 435, 511, 931]])
Transformer model:
 Transformer(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (multihead_attention): MultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (w_o): Linear(in_features=512, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): FeedForward(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
    